# Data exploration

## Downloading the Phase 7 data
Listing the files in the course dataset, downloading the Phase 7 files, and counting the cases in each file.



In [3]:
from google.colab import drive
drive.mount("/content/drive")

import json
from huggingface_hub import login, list_repo_files, hf_hub_download

# Logging in to Hugging Face to download the course dataset

login()

all_files = list_repo_files("Exploration-Lab/CS-781-Capstone", repo_type="dataset")

for rel_path in all_files:
    if rel_path.startswith("Phase_7") and rel_path.endswith(".json"):
        path = hf_hub_download(
            "Exploration-Lab/CS-781-Capstone",
            rel_path,
            repo_type="dataset",
            local_dir="/content/hf_cache_p7"
        )
        with open(path, encoding="utf-8") as f:
            data = json.load(f)

        first_id = list(data.keys())[0]
        print(rel_path, "| cases:", len(data), "| fields:", list(data[first_id].keys()))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Phase_7_PCR/Dev/dev.json | cases: 118 | fields: ['text', 'relevant_candidates']
Phase_7_PCR/Dev/dev_candidates.json | cases: 1023 | fields: ['text']
Phase_7_PCR/Test/test.json | cases: 237 | fields: ['text']
Phase_7_PCR/Test/test_candidates.json | cases: 1727 | fields: ['text']
Phase_7_PCR/Train/train.json | cases: 827 | fields: ['text', 'relevant_candidates']
Phase_7_PCR/Train/train_candidates.json | cases: 4320 | fields: ['text']


## Counting the relevant cases
Counting the relevant candidates per query in train and dev, and checking that they are all in the candidate pool of the same split.


In [4]:
DATA_DIR = "/content/hf_cache_p7/Phase_7_PCR"

for split, folder in [("train", "Train"), ("dev", "Dev")]:
    with open(DATA_DIR + "/" + folder + "/" + split + ".json", encoding="utf-8") as f:
        queries = json.load(f)
    with open(DATA_DIR + "/" + folder + "/" + split + "_candidates.json", encoding="utf-8") as f:
        candidates = json.load(f)

    gold_counts = []
    missing = 0
    for query_id in queries:
        gold_ids = queries[query_id]["relevant_candidates"]
        gold_counts.append(len(gold_ids))
        for gold_id in gold_ids:
            if gold_id not in candidates:
                missing += 1

    gold_counts.sort()
    print(split, "| min", gold_counts[0], "| median", gold_counts[len(gold_counts) // 2],
          "| max", gold_counts[-1], "| total", sum(gold_counts), "| not in pool", missing)


train | min 0 | median 4 | max 52 | total 5308 | not in pool 0
dev | min 1 | median 5 | max 92 | total 973 | not in pool 0


## Measuring case lengths
Measuring the length of every query and candidate case in Phase-1 tokens.


In [5]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("/content/drive/MyDrive/CS781/data_p1/tokenizer.json")

for rel_path in all_files:
    if rel_path.startswith("Phase_7") and rel_path.endswith(".json"):
        with open("/content/hf_cache_p7/" + rel_path, encoding="utf-8") as f:
            cases = json.load(f)

        lengths = []
        for case_id in cases:
            text = " ".join(cases[case_id]["text"])
            lengths.append(len(tokenizer.encode(text).ids))

        lengths.sort()
        print(rel_path, "| median", lengths[len(lengths) // 2],
              "| 90%", lengths[len(lengths) * 9 // 10], "| max", lengths[-1])


Phase_7_PCR/Dev/dev.json | median 5746 | 90% 20441 | max 132674
Phase_7_PCR/Dev/dev_candidates.json | median 5136 | 90% 21428 | max 518426
Phase_7_PCR/Test/test.json | median 5966 | 90% 17858 | max 114127
Phase_7_PCR/Test/test_candidates.json | median 4959 | 90% 17461 | max 518426
Phase_7_PCR/Train/train.json | median 5736 | 90% 15932 | max 149837
Phase_7_PCR/Train/train_candidates.json | median 4608 | 90% 15568 | max 518426


# Data preparation

## Making the chunks
Joining the sentences of every case, tokenizing them and keeping the first 4 chunks of 512 tokens, for the queries and candidates of each split.


In [7]:
import os
from huggingface_hub import get_token

os.environ["HF_TOKEN"] = get_token()

!python /content/drive/MyDrive/CS781/phase7/data_prep.py \
    --out_dir /content/drive/MyDrive/CS781/data_p7 \
    --tokenizer_path /content/drive/MyDrive/CS781/data_p1/tokenizer.json


Preparing train ...
  train queries: 817 cases kept, 10 skipped
  train candidates: 4320 cases kept, 0 skipped
Preparing dev ...
  dev queries: 118 cases kept, 0 skipped
  dev candidates: 1023 cases kept, 0 skipped
Preparing test ...
  test queries: 237 cases kept, 0 skipped
  test candidates: 1727 cases kept, 0 skipped

Saved to /content/drive/MyDrive/CS781/data_p7


# Training

## Run 1: decoder, full fine-tuning
Training all the weights of the Phase-1 decoder for retrieval, and keeping the checkpoint with the best dev Micro-F1@7.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

!python /content/drive/MyDrive/CS781/phase7/train.py \
    --arch decoder \
    --method full \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p7/decoder_full


Mounted at /content/drive
Using device: cuda
817 train queries, 8 queries per update, 206 steps total
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 1
Method: full, training 109.73M of 109.7M parameters (100.00%)
  before training, dev Micro-F1@7: 0.2090  precision 0.2276  recall 0.1932
step 10/206  epoch 1  loss 1.5165  elapsed 0.8 min
step 20/206  epoch 1  loss 1.6888  elapsed 1.6 min
step 30/206  epoch 1  loss 1.2519  elapsed 2.4 min
step 40/206  epoch 1  loss 1.4567  elapsed 3.2 min
step 50/206  epoch 1  loss 1.4895  elapsed 4.0 min
  dev Micro-F1@7: 0.1579  precision 0.1719  recall 0.1459
step 60/206  epoch 1  loss 1.4102  elapsed 5.7 min
step 70/206  epoch 1  loss 1.3206  elapsed 6.5 min
step 80/206  epoch 1  loss 1.2957  elapsed 7.3 min
step 90/206  epoch 1  loss 1.3961  elapsed 8.1 min
step 100/206  epoch 1  loss 1.3454  elapsed 8.9 min
  dev Micro-F1@7: 0.1790  precision 0.1949  recall 

# Prediction

## Run 1: checking on dev
Checking the best decoder full checkpoint on the dev queries and their candidate pool.



In [2]:
!python /content/drive/MyDrive/CS781/phase7/predict.py --split dev \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p7/decoder_full/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p7/decoder_full


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p7/decoder_full/best_checkpoint.pt (step 100, dev Micro-F1@7 0.1790)
118 queries, 0 with a bad list
dev Micro-F1@7: 0.1790  precision 0.1949  recall 0.1655
Saved /content/drive/MyDrive/CS781/submissions_p7/decoder_full/dev_ranked.json


## Run 1: making the Codabench file
Ranking the test candidates for every test query and saving prediction.json and submission.zip.


In [3]:
!python /content/drive/MyDrive/CS781/phase7/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p7/decoder_full/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p7/decoder_full


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p7/decoder_full/best_checkpoint.pt (step 100, dev Micro-F1@7 0.1790)
237 queries, 0 with a bad list
Saved /content/drive/MyDrive/CS781/submissions_p7/decoder_full/submission.zip


## Run 1 with hard negatives: decoder, full fine-tuning
Training the decoder with 4 hard negatives and 4 random negatives for every query, to see if harder wrong cases help retrieval.


In [5]:
from google.colab import drive
drive.mount("/content/drive")

!python /content/drive/MyDrive/CS781/phase7/train.py \
    --arch decoder \
    --method full \
    --num_hard 4 \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p7/decoder_full_hard


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
817 train queries, 8 queries per update, 206 steps total
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 1
Method: full, training 109.73M of 109.7M parameters (100.00%)
  before training, dev Micro-F1@7: 0.2090  precision 0.2276  recall 0.1932
  finding hard negatives ...
step 10/206  epoch 1  loss 2.9011  elapsed 4.5 min
step 20/206  epoch 1  loss 2.1755  elapsed 5.4 min
step 30/206  epoch 1  loss 2.1872  elapsed 6.2 min
step 40/206  epoch 1  loss 2.1926  elapsed 7.0 min
step 50/206  epoch 1  loss 2.1928  elapsed 7.8 min
  dev Micro-F1@7: 0.1757  precision 0.1913  recall 0.1624
step 60/206  epoch 1  loss 2.1918  elapsed 9.5 min
step 70/206  epoch 1  loss 2.1749  elapsed 10.3 min
step 80/206  epoch 1  loss 2.1528  elapsed 11.2 min
step 90/206  epoch 1

## Decoder full with a smaller learning rate for the last layer
Training the decoder again with the last layer at the same learning rate as the backbone, to see if the dev score stops falling.


In [4]:
from google.colab import drive
drive.mount("/content/drive")

!python /content/drive/MyDrive/CS781/phase7/train.py \
    --arch decoder \
    --method full \
    --head_learning_rate 3e-5 \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p7/decoder_full_lowhead


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
817 train queries, 8 queries per update, 206 steps total
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/decoder/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 1
Method: full, training 109.73M of 109.7M parameters (100.00%)
  before training, dev Micro-F1@7: 0.2090  precision 0.2276  recall 0.1932
step 10/206  epoch 1  loss 1.4196  elapsed 0.6 min
step 20/206  epoch 1  loss 1.3445  elapsed 1.3 min
step 30/206  epoch 1  loss 1.3382  elapsed 2.0 min
step 40/206  epoch 1  loss 1.3879  elapsed 2.6 min
step 50/206  epoch 1  loss 1.4627  elapsed 3.3 min
  dev Micro-F1@7: 0.1879  precision 0.2046  recall 0.1737
step 60/206  epoch 1  loss 1.3583  elapsed 4.6 min
step 70/206  epoch 1  loss 1.1956  elapsed 5.3 min
step 80/206  epoch 1  loss 1.1439  elapsed 6.0 min
step 90/206  epoch 1  loss 1.3351  elapsed 6.7 min


## Decoder full with smaller last layer learning rate: dev score
Getting the dev score from the best checkpoint.


In [5]:
!python /content/drive/MyDrive/CS781/phase7/predict.py --split dev \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p7/decoder_full_lowhead/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p7/decoder_full_lowhead


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p7/decoder_full_lowhead/best_checkpoint.pt (step 206, dev Micro-F1@7 0.2034)
118 queries, 0 with a bad list
dev Micro-F1@7: 0.2034  precision 0.2215  recall 0.1881
Saved /content/drive/MyDrive/CS781/submissions_p7/decoder_full_lowhead/dev_ranked.json


## Decoder full with smaller last layer learning rate: test file
Making the submission file for Codabench.


In [6]:
!python /content/drive/MyDrive/CS781/phase7/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p7/decoder_full_lowhead/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p7/decoder_full_lowhead


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p7/decoder_full_lowhead/best_checkpoint.pt (step 206, dev Micro-F1@7 0.2034)
237 queries, 0 with a bad list
Saved /content/drive/MyDrive/CS781/submissions_p7/decoder_full_lowhead/submission.zip


## Run 2: convmix, LoRMA
Training only the LoRMA weights and the last layer on the Phase-1 convmix.


In [6]:
!python /content/drive/MyDrive/CS781/phase7/train.py \
    --arch convmix \
    --method lorma \
    --learning_rate 1e-3 \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/convmix/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p7/convmix_lorma


Using device: cuda
817 train queries, 8 queries per update, 206 steps total
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/convmix/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 1
Method: lorma, training 0.69M of 105.5M parameters (0.65%)
  before training, dev Micro-F1@7: 0.1178  precision 0.1283  recall 0.1089
step 10/206  epoch 1  loss 1.8821  elapsed 0.7 min
step 20/206  epoch 1  loss 2.1101  elapsed 1.5 min
step 30/206  epoch 1  loss 2.1132  elapsed 2.2 min
step 40/206  epoch 1  loss 1.8160  elapsed 2.9 min
step 50/206  epoch 1  loss 1.7837  elapsed 3.6 min
  dev Micro-F1@7: 0.0767  precision 0.0835  recall 0.0709
step 60/206  epoch 1  loss 1.8152  elapsed 5.2 min
step 70/206  epoch 1  loss 1.5792  elapsed 6.0 min
step 80/206  epoch 1  loss 1.5389  elapsed 6.7 min
step 90/206  epoch 1  loss 1.6916  elapsed 7.4 min
step 100/206  epoch 1  loss 1.6534  elapsed 8.2 min
  dev Micro-F1@7: 0.0756  precision 0.0823  recall 0.0699
step 110/206  epoch 2 

## Run 2: checking on dev
Checking the best convmix LoRMA checkpoint on the dev queries and their candidate pool.


In [7]:
!python /content/drive/MyDrive/CS781/phase7/predict.py --split dev \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p7/convmix_lorma/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p7/convmix_lorma


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p7/convmix_lorma/best_checkpoint.pt (step 206, dev Micro-F1@7 0.0901)
118 queries, 0 with a bad list
dev Micro-F1@7: 0.0901  precision 0.0981  recall 0.0832
Saved /content/drive/MyDrive/CS781/submissions_p7/convmix_lorma/dev_ranked.json


## Run 2: making the Codabench file
Ranking the test candidates for every test query and saving prediction.json and submission.zip.


In [8]:
!python /content/drive/MyDrive/CS781/phase7/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p7/convmix_lorma/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p7/convmix_lorma


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p7/convmix_lorma/best_checkpoint.pt (step 206, dev Micro-F1@7 0.0901)
237 queries, 0 with a bad list
Saved /content/drive/MyDrive/CS781/submissions_p7/convmix_lorma/submission.zip


## Run 3: moe, partial fine-tuning
Training only the last 4 layers, the final norm and the last layer on the Phase-1 moe.


In [9]:
!python /content/drive/MyDrive/CS781/phase7/train.py \
    --arch moe \
    --method partial \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/moe/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p7/moe_partial


Using device: cuda
817 train queries, 8 queries per update, 206 steps total
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/moe/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 1
Method: partial, training 69.66M of 194.4M parameters (35.84%)
  before training, dev Micro-F1@7: 0.1123  precision 0.1223  recall 0.1038
step 10/206  epoch 1  loss 1.9003  elapsed 0.6 min
step 20/206  epoch 1  loss 2.0886  elapsed 1.2 min
step 30/206  epoch 1  loss 2.0848  elapsed 1.8 min
step 40/206  epoch 1  loss 2.0170  elapsed 2.4 min
step 50/206  epoch 1  loss 1.9502  elapsed 2.9 min
  dev Micro-F1@7: 0.0678  precision 0.0738  recall 0.0627
step 60/206  epoch 1  loss 1.9390  elapsed 4.3 min
step 70/206  epoch 1  loss 1.8558  elapsed 4.9 min
step 80/206  epoch 1  loss 1.9166  elapsed 5.4 min
step 90/206  epoch 1  loss 2.0043  elapsed 6.0 min
step 100/206  epoch 1  loss 1.9457  elapsed 6.6 min
  dev Micro-F1@7: 0.0678  precision 0.0738  recall 0.0627
step 110/206  epoch 2 

## Run 3: checking on dev
Checking the best moe partial checkpoint on the dev queries and their candidate pool.


In [10]:
!python /content/drive/MyDrive/CS781/phase7/predict.py --split dev \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p7/moe_partial/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p7/moe_partial


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p7/moe_partial/best_checkpoint.pt (step 50, dev Micro-F1@7 0.0678)
118 queries, 0 with a bad list
dev Micro-F1@7: 0.0678  precision 0.0738  recall 0.0627
Saved /content/drive/MyDrive/CS781/submissions_p7/moe_partial/dev_ranked.json


## Run 3: making the Codabench file
Ranking the test candidates for every test query and saving prediction.json and submission.zip.


In [11]:
!python /content/drive/MyDrive/CS781/phase7/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p7/moe_partial/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p7/moe_partial


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p7/moe_partial/best_checkpoint.pt (step 50, dev Micro-F1@7 0.0678)
237 queries, 0 with a bad list
Saved /content/drive/MyDrive/CS781/submissions_p7/moe_partial/submission.zip


## Moe partial with a smaller learning rate for the last layer
Training moe again with the last layer at the same learning rate as the backbone.


In [7]:
from google.colab import drive
drive.mount("/content/drive")

!python /content/drive/MyDrive/CS781/phase7/train.py \
    --arch moe \
    --method partial \
    --head_learning_rate 3e-5 \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p1/moe/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p7/moe_partial_lowhead


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
817 train queries, 8 queries per update, 206 steps total
Loading Phase-1 checkpoint: /content/drive/MyDrive/CS781/ckpt_p1/moe/best_checkpoint.pt
Backbone loaded, missing keys: 0, unused keys: 1
Method: partial, training 69.66M of 194.4M parameters (35.84%)
  before training, dev Micro-F1@7: 0.1123  precision 0.1223  recall 0.1038
step 10/206  epoch 1  loss 1.8147  elapsed 0.5 min
step 20/206  epoch 1  loss 1.9414  elapsed 1.0 min
step 30/206  epoch 1  loss 1.9249  elapsed 1.5 min
step 40/206  epoch 1  loss 1.9764  elapsed 2.1 min
step 50/206  epoch 1  loss 1.9904  elapsed 2.6 min
  dev Micro-F1@7: 0.1178  precision 0.1283  recall 0.1089
step 60/206  epoch 1  loss 2.0003  elapsed 4.0 min
step 70/206  epoch 1  loss 1.9024  elapsed 4.5 min
step 80/206  epoch 1  loss 1.9258  elapsed 5.0 min
step 90/206  epoch 1  loss 1.9816  elapsed 5.6 min
ste

## Moe partial with smaller last layer learning rate: dev score
Getting the dev score from the best checkpoint.


In [8]:
!python /content/drive/MyDrive/CS781/phase7/predict.py --split dev \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p7/moe_partial_lowhead/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p7/moe_partial_lowhead


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p7/moe_partial_lowhead/best_checkpoint.pt (step 50, dev Micro-F1@7 0.1178)
118 queries, 0 with a bad list
dev Micro-F1@7: 0.1178  precision 0.1283  recall 0.1089
Saved /content/drive/MyDrive/CS781/submissions_p7/moe_partial_lowhead/dev_ranked.json


## Moe partial with smaller last layer learning rate: test file
Making the submission file for Codabench.


In [9]:
!python /content/drive/MyDrive/CS781/phase7/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p7/moe_partial_lowhead/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p7/moe_partial_lowhead


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p7/moe_partial_lowhead/best_checkpoint.pt (step 50, dev Micro-F1@7 0.1178)
237 queries, 0 with a bad list
Saved /content/drive/MyDrive/CS781/submissions_p7/moe_partial_lowhead/submission.zip


## Run 4: encdec, knowledge distillation
Training the encoder half of the Phase-1 encdec as a smaller student that learns from the relevant candidates and the decoder full teacher.


In [12]:
!python /content/drive/MyDrive/CS781/phase7/distill.py \
    --teacher_ckpt /content/drive/MyDrive/CS781/ckpt_p7/decoder_full/best_checkpoint.pt \
    --p1_ckpt /content/drive/MyDrive/CS781/ckpt_p1/encdec/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/ckpt_p7/encdec_distill


Using device: cuda
Student (encdec) backbone loaded, missing keys: 0, unused keys: 68
Teacher: 109.7M parameters, student: 58.9M parameters (54% of the teacher)
817 train queries, 8 queries per update, 206 steps total
  before training, dev Micro-F1@7: 0.0278  precision 0.0303  recall 0.0257
step 10/206  epoch 1  loss 1.7314  elapsed 1.0 min
step 20/206  epoch 1  loss 1.7724  elapsed 2.0 min
step 30/206  epoch 1  loss 1.7815  elapsed 2.9 min
step 40/206  epoch 1  loss 1.7629  elapsed 3.9 min
step 50/206  epoch 1  loss 1.7166  elapsed 4.9 min
  dev Micro-F1@7: 0.0311  precision 0.0339  recall 0.0288
step 60/206  epoch 1  loss 1.7313  elapsed 6.3 min
step 70/206  epoch 1  loss 1.6516  elapsed 7.2 min
step 80/206  epoch 1  loss 1.5977  elapsed 8.2 min
step 90/206  epoch 1  loss 1.5965  elapsed 9.2 min
step 100/206  epoch 1  loss 1.4997  elapsed 10.2 min
  dev Micro-F1@7: 0.0267  precision 0.0291  recall 0.0247
step 110/206  epoch 2  loss 1.4953  elapsed 11.4 min
step 120/206  epoch 2  los

## Run 4: checking on dev
Checking the best encdec student checkpoint on the dev queries and their candidate pool.


In [13]:
!python /content/drive/MyDrive/CS781/phase7/predict.py --split dev \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p7/encdec_distill/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p7/encdec_distill


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p7/encdec_distill/best_checkpoint.pt (step 150, dev Micro-F1@7 0.0389)
118 queries, 0 with a bad list
dev Micro-F1@7: 0.0389  precision 0.0424  recall 0.0360
Saved /content/drive/MyDrive/CS781/submissions_p7/encdec_distill/dev_ranked.json


## Run 4: making the Codabench file
Ranking the test candidates for every test query and saving prediction.json and submission.zip.


In [14]:
!python /content/drive/MyDrive/CS781/phase7/predict.py --split test \
    --ckpt /content/drive/MyDrive/CS781/ckpt_p7/encdec_distill/best_checkpoint.pt \
    --data_dir /content/drive/MyDrive/CS781/data_p7 \
    --out_dir /content/drive/MyDrive/CS781/submissions_p7/encdec_distill


Using device: cuda
Loaded /content/drive/MyDrive/CS781/ckpt_p7/encdec_distill/best_checkpoint.pt (step 150, dev Micro-F1@7 0.0389)
237 queries, 0 with a bad list
Saved /content/drive/MyDrive/CS781/submissions_p7/encdec_distill/submission.zip
